# Marconi daily DEM movie
csherwood@usgs.gov, 2026-10-26, generated with Claude Opus 5.5

Regenerates a five-panel frame for each day from the GeoTIFFs written by `make_dems.ipynb`, then assembles the
frames into a Motion-JPEG .avi:
* a, b: $H_s$ and water level at the nanopod (static), with a red bar at the DEM day and its three-day window shaded;
* c: DEM with labeled 1 m contours (date above);
* d: change from the previous non-empty DEM (blue accretion, red erosion; "minus" that date above);
* e: uncertainty.

Panels c–e share the same scale and bounds (`MOVIE['xlim']`, `MOVIE['ylim']`; None = data extent).

Requires: numpy, pandas, matplotlib, opencv, rasterio.

In [1]:
from pathlib import Path
import pandas as pd
import marconi_movie as mm

# --- config
PROJ = Path(r'F:\crs\proj\2026_wl_dems')
WAVES = PROJ / 'nanopod_waves' / 'sig1000_waves_ALL1.csv'
OUT_DIR = PROJ / 'scan'
DEM_DIR = OUT_DIR / 'dems'
FIG_DIR = PROJ / 'fig'
FRAME_DIR = FIG_DIR / 'movie_frames'
AVI = FIG_DIR / 'marconi_dems.avi'
FPS = 4
FIRST_DAY, LAST_DAY = None, None     # e.g. '2024-12-10', '2025-02-15'; None = all days in dem_days.csv
MOVIE = mm.MOVIE | {'xlim': (None, 420200.)}

FRAME_DIR.mkdir(parents=True, exist_ok=True)
waves = pd.read_csv(WAVES, parse_dates=['time'])
dt = pd.read_csv(OUT_DIR / 'dem_days.csv', parse_dates=['day'])
dt = dt[(dt.day >= pd.Timestamp(FIRST_DAY or dt.day.min())) & (dt.day <= pd.Timestamp(LAST_DAY or dt.day.max()))]
status = {r.day: f"{r.status}, {r.n_img_used:.0f} images, slope {r.slope_median:.3f}" if r.status == 'ok' else r.status
          for r in dt.itertuples()}
print(len(dt), 'days;', mm.data_bounds(DEM_DIR, dt.day, MOVIE))

92 days; ([np.float64(420071.0), 420200.0], [np.float64(4638301.0), np.float64(4638683.0)])


In [2]:
paths = mm.make_frames(DEM_DIR, FRAME_DIR, list(dt.day), waves, status, MOVIE)
print(f'{len(paths)} frames in {FRAME_DIR}')


92 frames in F:\crs\proj\2026_wl_dems\fig\movie_frames


## Movie
Assembles all `frame_####.png` in `FRAME_DIR` into the .avi; rerun alone to change `FPS` without redrawing frames.


In [5]:
FPS=1
frames = sorted(FRAME_DIR.glob('frame_*.png'))
mm.write_avi(frames, AVI, FPS)
print(f'{len(frames)} frames at {FPS} per s -> {AVI} ({AVI.stat().st_size / 1e6:.1f} MB)')


92 frames at 1 per s -> F:\crs\proj\2026_wl_dems\fig\marconi_dems.avi (19.8 MB)
